# 02 Feature Engineering: Key Takeaways

**Purpose:** Build the 5 engineered features that turn cleaned race data into the proxies needed to answer the research question: driver, strategy, or both, and has the balance shifted across eras?

## The Five Features at a Glance

**1. position_change** (driver skill, race grain, lives on results)
Grid minus finishing position. Positive = gained places during the race. The grid already prices in the car, so movement after lights out points to the driver.
Key numbers: 1,039 DNF rows stay null (no finish = undefined). 85 pit lane starts (grid = 0 is a code, not a position) set to null with a .loc mask. Roughly 5,300 of 6,436 rows usable.

**2. era** (story structure, race grain, lives on results)
Year binned into the four eras with pd.cut. Bins [2009, 2013, 2020, 2021, 2024], first edge at 2009 so 2010 is not silently dropped.
Key numbers: era counts 1,810 / 2,827 / 440 / 1,359 (sums to 6,436). 2021 is one season, so its small sample matters for statistical power later.

**3. points_per_race** (driver skill, driver-season grain, new table driver_season)
Total grand prix points divided by races started, per driver per season. Fairness metric: 120 points from 15 races is not the same season as 120 from 22.
Key numbers: 42 never-started rows excluded from the denominator (37 W, 4 F, 1 E). 349 driver-seasons. Spot check: Vettel 2013 = 397 pts, 19 races, 20.89 per race, matches reality.

**4. teammate_gap** (driver skill, strongest proxy, team-season grain, new table team_season)
Same car, two drivers: higher scorer's season points minus lower scorer's. The car stops being a variable.
Key numbers: 159 team-seasons. 35 of them had 3+ drivers from mid-season swaps, rule = keep top two by races entered, points as tiebreaker. Spot check: Mercedes 2016 gap = exactly 5.0 (Rosberg 385, Hamilton 380).

**5. avg_pit_stop_per_team_per_season** (team strategy, team-season grain, new table avg_pit_stop)
Mean pit stop duration_seconds per constructor per season, computed on crew service events only.
Key numbers: 147 rows = 159 team-seasons minus the 12 teams of 2010 (pit data starts 2011). 517 stops of 60+ seconds excluded by deliberate rule as race stoppage parking events, not crew service (they are red flag or safety stoppages, median ~22 minutes). duration_seconds is built from milliseconds/1000 in the cleaning notebook, not parsed from the duration string. Full pit lane transit (~24s average), not the 2-3s tire change, so cross-season second comparisons carry a calendar caveat.

## Decisions That Apply Across Features
1. No imputation ever: if a real value does not exist, the cell stays null. Nulls sit out of the math, fake numbers poison it.
2. Sprint points excluded everywhere: grand prix points only, so all four eras are measured on the identical scoring source. 2021-2024 totals will not match official standings, documented tradeoff.
3. Every feature verified against a real-world fact before moving on (Vettel 2013, Mercedes 2016, McLaren 2012 gap of 2, Red Bull pit crew era).
4. Three grains now exist: results (race), driver_season (driver-season), team_season and avg_pit_stop (team-season). Each table keeps one grain.
5. Pit stop correction (documented Discernment finding): the 517 "nulls" originally attributed to red flag races were actually parse failures on MM:SS.mmm strings for stops of 60+ seconds. Corrected by rebuilding duration_seconds from milliseconds in cleaning, then applying a deliberate 60s exclusion rule in feature 5. External AI review flagged the mechanism, verified against raw data, remedy (include the long stops) rejected on evidence.

In [1]:
import pandas as pd

In [2]:
# load in all the cleaned dataframes from the clean folder
results = pd.read_csv('../data/clean/results.csv')
races = pd.read_csv('../data/clean/races.csv')
pit_stops = pd.read_csv('../data/clean/pit_stops.csv')
drivers = pd.read_csv('../data/clean/drivers.csv')  
constructors = pd.read_csv('../data/clean/constructors.csv')

## Setup: Load Cleaned Data and Recast Nullable Dtypes

This notebook builds the 5 engineered features on top of the cleaned CSVs exported from `01_data_cleaning.ipynb`. Setup runs in five short steps.

### 1. Load
All 5 cleaned core files are read from `data/clean/`: results, races, pit_stops, drivers, constructors.

### 2. Why recast on load
CSV format does not preserve nullable Int64 metadata. On write, pandas exports the column as strings that pandas re-reads as float64 (because int64 cannot hold nulls). Any downstream operation that expects whole numbers or Int64 semantics will misbehave unless the column is explicitly recast. This step reverses that CSV round-trip.

### 3. What gets recast
Two columns need explicit recasting:
- `results.position` -> Int64 (nullable integer, DNF nulls preserved)
- `drivers.number` -> Int64 (nullable integer, pre-2014 nulls preserved)

`pit_stops.duration_seconds` is NOT in dtypes_dict. It was born float64 in the cleaning notebook by dividing `milliseconds / 1000`, and CSV export/reload preserves float64 correctly. No declaration needed.

`pit_stops.duration` (the original string column) stays a string. It is kept as human-readable evidence, not used for math.

### 4. The dict-and-loop pattern
Reuses the same dictionary-plus-loop pattern from the cleaning notebook. Adding or removing a column means editing one dictionary entry, not hunting through scattered code. Forward-compatible with the Week 5 refactor into `.py` pipeline scripts.

### 5. Reassignment
After the astype loop, the recast DataFrames are reassigned back to the standalone variables so downstream cells work with the corrected types.

In [3]:
# create a dictionary of the dataframes
dataframes = {
    'results': results,
    'races': races,
    'pit_stops': pit_stops,
    'drivers': drivers,
    'constructors': constructors
}

In [4]:
# Build the dtypes dictionary for astype recasting. 
# Only position and number need recasting: CSV does not preserve nullable Int64 metadata.
# pit_stops.duration_seconds is born float64 from milliseconds/1000 in cleaning,
# so it survives the CSV round-trip and needs no declaration.
dtypes_dict = {
    'results': {
        'position': 'Int64'
    },
    'drivers': {
        'number': 'Int64'
    }
}

In [5]:
for name, cols in dtypes_dict.items():
    dataframes[name] = dataframes[name].astype(cols)

In [6]:
# Reassign the recast DataFrames back to the standalone variables
results = dataframes['results']
drivers = dataframes['drivers']
pit_stops = dataframes['pit_stops']

## Feature 1: position_change

**Formula:** `position_change = grid - position`

**Type:** Driver skill proxy (primary)

### What It Measures
The number of positions a driver gained or lost during the race. Positive means the driver gained places (started 10th, finished 6th gives +4). Negative means the driver lost places. Zero means they finished where they started.

### Why It Is a Driver Skill Proxy
The starting grid already prices in the car. Qualifying sorts the field by car pace, so by the time the lights go out, car speed is baked into the grid slot. Any movement after that points to what the driver did on Sunday: overtakes, defending, tire management, racecraft.

### Null Decision 1: DNFs Stay Null
A driver who did not finish has no finishing position, so positions gained is undefined for them. `grid - position` propagates the null automatically (any math with pd.NA returns pd.NA), so no special handling was needed. Assigning a 0 instead would claim "crashed on lap 2" and "held position all race" are the same performance, and would violate the no-imputation principle from the cleaning notebook.

### Null Decision 2: Pit Lane Starts (grid = 0) Set to Null
In this dataset, grid = 0 is a code for a pit lane start, not a real grid slot. Left alone, the formula produces upside-down results: a driver who started from the pit lane (behind everyone) and climbed to P15 would show -15, claiming they collapsed when they actually gained places. 85 rows (about 1.3% of results) were affected.

Fix applied with a vectorized boolean mask, no loops:

`results.loc[results['grid'] == 0, 'position_change'] = pd.NA`

Justification: grid = 0 is a code, not a position, so position_change is undefined for pit lane starts. Inventing a starting position (for example treating them as P20) would be imputation.

### Verification: Three Checks, Three Claims
Each null decision gets its own proof, and when two mechanisms can overlap on the same rows, a check is added to separate them.

1. DNF check (position is null): proves null propagation worked for drivers who started normally but did not finish. These rows have real grid numbers, so grid = 0 is not involved.
2. All grid = 0 rows: proves the .loc fix ran. On its own this check has a blind spot, because pit lane starters who also DNFed were already null from propagation before the fix ran.
3. Grid = 0 AND position is not null: closes the blind spot. These are pit lane starters who finished the race, so propagation could not have made them null. The only way these rows show NA is if the .loc fix did its job. This check isolates the fix as the cause.

### What the Nulls Mean Downstream
The row stays, one cell sits out. A pit lane starter or DNF row still contributes its points to points_per_race and teammate_gap, its statusId to the DNF story, and its pit stops to avg_pit_stop_per_team_per_season. Only position_change is benched for that row. Pandas aggregations (.mean(), .corr(), groupby) skip nulls by default, so era comparisons and correlations are computed only on honest values, roughly 5,300+ usable rows out of 6,436.

### Limitation
position_change only measures finished races from a numbered grid slot. A driver who crashes frequently looks fine on this metric because their DNFs vanish from it. This is covered by pairing it with teammate_gap, which is built on points, where a DNF scores zero for real. Team pit strategy calls can also gain or lose positions, so this proxy is not purely driver skill. Documented tradeoff, used alongside other proxies.

In [7]:
# create position_change
results['position_change'] = results['grid'] - results['position']

In [8]:
(results['grid'] == 0).sum()

85

In [9]:
results.loc[results['grid'] == 0, 'position_change'] = pd.NA

In [10]:
results[results['grid'] == 0][['grid', 'position', 'position_change']].head()

,grid,position,position_change
931,0,<NA>,<NA>
932,0,<NA>,<NA>
1100,0,<NA>,<NA>
2231,0,<NA>,<NA>
2251,0,<NA>,<NA>


In [11]:
# Verify null propagation for DNF rows
results[results['position'].isna()][['grid', 'position', 'position_change']].head()

,grid,position,position_change
17,14,<NA>,<NA>
18,23,<NA>,<NA>
19,19,<NA>,<NA>
20,17,<NA>,<NA>
21,16,<NA>,<NA>


In [12]:
results[(results['grid'] == 0) & (results['position'].notna())][['grid', 'position', 'position_change']].head()

,grid,position,position_change
2321,0,10,<NA>
2787,0,20,<NA>
3926,0,10,<NA>
3946,0,10,<NA>
3952,0,16,<NA>


## Feature 2: era

**Formula:** `pd.cut()` binning `year` into four labeled eras

**Type:** Story structure and grouping variable

### What It Is
A categorical label assigning every result row to one of the four eras that structure the entire analysis: 2010-2013 (Vettel and Red Bull), 2014-2020 (Hamilton and Mercedes), 2021 (Verstappen vs Hamilton, closest fight), 2022-2024 (Verstappen and Red Bull again). Era comparisons are the backbone of the research question: has the balance between driver and strategy shifted over time?

### Prerequisite: Merging year onto results
`year` lives in races, not results. The two tables share `raceId`, so year was borrowed with a targeted merge:

`results = results.merge(races[['raceId', 'year']], on='raceId')`

Only the two needed columns were pulled from races, not the whole table. This keeps results lean and avoids column collisions. Verified after the merge: row count unchanged at 6,436, confirming a one-to-one join with no duplication or drops.

### Why pd.cut
Bucketing years into eras is exactly what `pd.cut()` is built for. It is vectorized (all rows stamped in one pass), maintainable (era definitions live in one bins list and one labels list, so redrawing a boundary means changing one number in one place), and self-documenting. Alternatives like row loops or stacked .loc masks are slower and scatter the boundary logic across multiple lines.

### The Fence Post Logic
Four eras require five bin edges: `[2009, 2013, 2020, 2021, 2024]`. Each bin excludes its left edge and includes its right edge. The first post sits at 2009, not 2010, because a bin starting at 2010 would exclude 2010 itself and silently null out the entire first season. Each remaining post is the final year of an era.

### Verification
Grouped by era and checked min year, max year, and count per era. Every era's min and max land exactly on its boundary years, 2021 contains only 2021, the four counts sum to 6,436, and a null check on era returned 0. No years leaked between eras and no rows fell outside the bins.

### Sample Size Note for Statistical Testing
Era sample sizes are structurally imbalanced because the eras span different numbers of seasons: 2010-2013 has 1,810 results across 4 seasons, 2014-2020 has 2,827 across 7 seasons, 2021 has 440 from its single season, and 2022-2024 has 1,359 across 3 seasons. This is a property of the era definitions, not a data quality issue. Flagged now because the 2021 era's small sample will affect statistical power when running t-tests and other comparisons across eras in the EDA and statistical testing phase.

In [13]:
# Merge the year from the races into the results dataframe
results = results.merge(races[['raceId', 'year']], on='raceId')

In [14]:
# Every year falls into exactly one of four bins. 2010-2013, 2014-2020, 2021, 2022-2024
# Create era label by binning year into the four eras
results['era'] = pd.cut(
    results['year'],
    bins=[2009, 2013, 2020, 2021, 2024],
    labels=['2010-2013', '2014-2020', '2021', '2022-2024']
)

In [15]:
results.groupby('era', observed=True)['year'].agg(['min', 'max', 'count'])

,min,max,count
era,,,
2010-2013,2010,2013,1810
2014-2020,2014,2020,2827
2021,2021,2021,440
2022-2024,2022,2024,1359


In [16]:
results['era'].isna().sum()

0

## Feature 3: points_per_race

**Formula:** total grand prix points divided by races started, per driver per season

**Type:** Driver skill proxy (fairness-adjusted points measure)

### What It Measures
A driver's scoring rate per season. Raw season totals are unfair to drivers who missed races through injury or mid-season swaps: 120 points from 22 races is not the same season as 120 points from 15. Dividing by races started puts every driver-season on the same scale.

### Grain Change
This feature does not live on results. It lives in a new table, driver_season, at one row per driver per season, built with a groupby aggregation. results remains at one row per driver per race.

### Denominator Decision: Races Started, Not Entries
Rows where the driver never took the start are excluded before aggregation: W (withdrew, 37 rows), F (failed to qualify, 4 rows), E (excluded, 1 row), 42 rows total. A driver who never started had no opportunity to score, so counting those as races would deflate their average for events that never happened. R (retired), D (disqualified), and N (not classified) rows stay in the denominator because those drivers raced, and scoring zero from a start is honest signal. Races started is counted with nunique on raceId to guard against duplicate rows.

### Sprint Points Exclusion (Design Decision from Data Audit)
points_per_race is computed from grand prix points only. Sprint race points (introduced 2021, stored in sprint_results.csv) are excluded so all four eras are measured on the identical scoring source. Sprints exist in only 4 of the 15 seasons, so including them would change what is being measured partway through the timeline of a cross-era comparison. Consequence: season totals for 2021-2024 will differ slightly from official championship standings (18 sprint points awarded in 2021, 108 in 2022, 216 in 2023, 216 in 2024). Measurement consistency across eras was prioritized over matching official totals.

### Known Points Quirks Inside the Window (Documentation Only)
The 2014 Abu Dhabi finale awarded double points (a maximum of 50 for the win, one season only). A fastest lap bonus point existed from 2019 through 2024 (maximum 26 per race in those years). Both are left in the data as real championship points under the rules of their day, and noted here so reviewers are not surprised by per-race values above 25.

### Verification
Row count matches one row per driver per season. No nulls in the output table. Spot check against a known season (Vettel 2013) matches real-world championship records.

In [17]:
# Exclude rows where the driver never started: W (withdrew), F (failed to qualify), E (excluded)
# Denominator counts races started, not entry list appearances
# keep every row where positionText is not W, F, or E.
results_started = results[~results['positionText'].isin(['W', 'F', 'E'])].copy()

# Sanity check: 6436 - 42 = 6394
results_started.shape

(6394, 11)

In [18]:
# Group by driver and season, compute total points and races started per pile
# how many different races did this driver take part in that season?
driver_season = (
    results_started
    .groupby(['driverId', 'year'])
    .agg(
        total_points=('points', 'sum'),
        races_started=('raceId', 'nunique')
    )
    .reset_index()
)

# The feature itself
driver_season['points_per_race'] = driver_season['total_points'] / driver_season['races_started']

driver_season.head()

,driverId,year,total_points,races_started,points_per_race
0,1,2010,240.0,19,12.631579
1,1,2011,227.0,19,11.947368
2,1,2012,190.0,20,9.500000
3,1,2013,189.0,19,9.947368
4,1,2014,384.0,19,20.210526


In [19]:
# Check 1: shape. One row per driver per season, expect roughly 350-400 rows
print(driver_season.shape)

# Check 2: no nulls anywhere in the new table
print(driver_season.isna().sum())

# Check 3: spot check a known season. Vettel 2013 (driverId 20): dominant year, should be around 397 points over 19 races
driver_season[(driver_season['driverId'] == 20) & (driver_season['year'] == 2013)]

(349, 5)
driverId           0
year               0
total_points       0
races_started      0
points_per_race    0
dtype: int64


,driverId,year,total_points,races_started,points_per_race
86,20,2013,397.0,19,20.894737


## Feature 4: teammate_gap

**Formula:** higher scorer's season points minus lower scorer's season points, per team per season

**Type:** Driver skill proxy (strongest)

### What It Measures
The points spread between the two drivers of the same team in a season. Both drivers have identical machinery, so a consistent gap between them is the cleanest signal of driver skill in the data. The car is no longer a variable.

### Grain
One row per team per season, stored in a new table, team_season (159 rows). This is a third grain level in the project: results is race grain, driver_season is driver-season grain, team_season is team-season grain. Each table keeps one consistent grain.

### The Three-Step Build
1. Pile table: group results_started by constructorId, driverId, and year, aggregating each driver's season_points (sum of points) and races_entered (nunique of raceId). Built from results_started so withdrawn (W), failed to qualify (F), and excluded (E) entries stay out, consistent with Feature 3.
2. Top-two eviction: 35 of 159 team-seasons (about 22%) ran 3 or more drivers due to mid-season swaps, even though every team fields exactly 2 cars per race. Rule: keep the top two drivers by races_entered per team-season. A substitute who ran a handful of races never had a season-length opportunity, so including them would distort a comparison built on sustained exposure to the same car. Implemented by sorting on races_entered (descending) then taking head(2) per team-season group.
3. The gap: group the surviving pairs by constructorId and year, take max and min of season_points, subtract. Because each group holds exactly two drivers, max and min are simply the two teammates' totals.

### Tiebreaker Decision
The sort uses season_points as a secondary key. If three drivers tie on races entered, the choice of which two survive would otherwise depend on arbitrary row order. Ties go to the higher scorer, making the pipeline deterministic and reproducible.

### Sprint Points Exclusion
Grand prix points only, same design decision as Feature 3 (see that section for the full reasoning). All four eras are measured on the identical scoring source.

### Verification
Row count is exactly 159, matching the independently counted number of team-seasons in the window. Max group size after eviction is 2, confirming no team-season kept a third driver. Spot check against a known season: Mercedes 2016 returns points_high 385, points_low 380, gap 5.0, exactly matching the real Rosberg vs Hamilton title fight. No nulls in the output. No negative gaps, which is structurally guaranteed by the max minus min construction.

### Limitation
The gap measures spread, not direction or cause. A gap of 100 cannot distinguish "one exceptional driver" from "one underperforming driver," and it cannot see whether the stronger teammate was favored by strategy calls. It also inherits the DNF asymmetry: a driver's mechanical failures lower their points through no fault of their own. Used alongside position_change and interpreted per era rather than as a standalone verdict.

In [20]:
# teammate_gap measures: the points difference between the two drivers of the same team in the same season
team_driver_season = (
    results_started
    .groupby(['constructorId', 'driverId', 'year'])
    .agg(
        season_points=('points', 'sum'),
        races_entered=('raceId', 'nunique')
    )
    .reset_index()
)

team_driver_season.head(10)

,constructorId,driverId,year,season_points,races_entered
0,1,1,2010,240.0,19
1,1,1,2011,227.0,19
2,1,1,2012,190.0,20
3,1,4,2015,11.0,18
4,1,4,2016,54.0,20
5,1,4,2017,17.0,18
6,1,4,2018,50.0,21
7,1,18,2010,214.0,19
8,1,18,2011,270.0,19
9,1,18,2012,188.0,20


In [21]:
top_two = (
    team_driver_season
    .sort_values(['races_entered', 'season_points'], ascending=False)
    .groupby(['constructorId', 'year'])
    .head(2)
)

top_two[(top_two['constructorId'] == 9) & (top_two['year'] == 2019)]

,constructorId,driverId,year,season_points,races_entered
160,9,830,2019,278.0,21
166,9,842,2019,63.0,12


In [22]:
# For each team-season, the gap is the higher scorer minus the lower scorer
team_season = (
    top_two
    .groupby(['constructorId', 'year'])
    .agg(
        points_high=('season_points', 'max'),
        points_low=('season_points', 'min')
    )
    .reset_index()
)

# The feature itself
team_season['teammate_gap'] = team_season['points_high'] - team_season['points_low']

team_season.head()

,constructorId,year,points_high,points_low,teammate_gap
0,1,2010,240.0,214.0,26.0
1,1,2011,270.0,227.0,43.0
2,1,2012,190.0,188.0,2.0
3,1,2013,73.0,49.0,24.0
4,1,2014,126.0,55.0,71.0


## Feature 5: avg_pit_stop_per_team_per_season

**Formula:** mean of `duration_seconds`, per constructor per season, computed on crew service events only (`duration_seconds < 60`)

**Type:** Team strategy proxy (primary)

### What It Measures
Average pit stop duration for each team in each season, in seconds. Pit stop execution is purely the team's contribution to race outcomes since the driver is stationary while it happens. This is the strategy side of the research question, meeting the driver proxies in the era comparison.

### Build
`pit_stops` has no `constructorId` or `year`, so both were borrowed from `results` in a single merge. The join requires BOTH keys, `raceId` and `driverId` together: `raceId` alone matches roughly 20 drivers per race and `driverId` alone matches one driver across hundreds of races, either of which would explode the row count. Together they pinpoint exactly one results row per pit stop. `how='left'` with a null check afterward turns the merge into a test of the claim that every pit stop belongs to a race entry. Verified: shape unchanged in rows, `constructorId` nulls 0.

The numeric column used is `duration_seconds`, built in the cleaning notebook from `milliseconds / 1000` rather than from parsing the `duration` string. See the Pit Stop Correction note in the cleaning notebook for the full audit trail.

### Deliberate Exclusion: Stops of 60+ Seconds
517 pit stop rows have `duration_seconds` values of 60 seconds or more, with a median of roughly 22 minutes and a maximum of roughly 51 minutes. These are race stoppage parking events (red flags and safety stoppages), not crew service. Including them would shift team-season averages by up to 225 seconds, which would destroy the crew execution proxy the feature is designed to be.

Applied as a documented rule via a boolean mask:

`pit_stops_service = pit_stops[pit_stops['duration_seconds'] < 60].copy()`

Result: 11,371 stops filtered to 10,854 crew service events. The mean is then computed on this filtered set.

### Design Decision: Raw Average vs Within-Race Relative Measure
`duration_seconds` is full pit lane transit time, entry to exit (mean roughly 24 seconds), not just the 2 to 3 second tire change. Pit lane length varies by circuit, so raw averages partly reflect the season's calendar. Decision: use the raw average anyway. Within a season, every team runs the same calendar, so the circuit mix affects all teams roughly equally and team-vs-team comparison stays fair, which is where the analysis lives. It also keeps the feature in units a race strategist actually uses. Caveat: cross-season comparisons of absolute seconds reflect calendar changes as well as crew performance, so season-level rankings are the safer read across years. A within-race relative measure (each stop minus the race median) would cancel the circuit effect and is noted as future work.

### Known Gap: 2010
Pit stop data in this dataset starts in 2011. The 2010 team-seasons are absent from this table entirely: 147 rows instead of the 159 team-seasons in `teammate_gap`, a difference of exactly the 12 teams that raced in 2010. Documented limitation carried from Week 1.

### Verification
Shape (147, 3) matches the predicted 159 minus 12. Zero rows for 2010. Zero nulls in `avg_pit_stop_duration` (`duration_seconds` is fully populated, no null skipping happening at the mean step). Row count before the exclusion: 11,371. Row count after: 10,854. Difference of exactly 517, matching the count of long stops identified in the pit stop correction. Eyeball check: the top pit crews (Ferrari, Mercedes, Red Bull) cluster at the fast end of 2023, consistent with real-world pit crew standings.

### Limitation
Averages measure execution speed only, not strategy quality. When a team chose to pit (undercut, safety car call) is the other half of strategy, partially captured by the `lap` column and noted for EDA. Cross-season absolute comparisons inherit the calendar caveat above.

In [23]:
# Borrow constructorId from results so pit stops know which team they belong to
# Join on BOTH keys: raceId alone matches ~20 drivers per race, driverId alone
# matches one driver across hundreds of races. Together they pinpoint one row.
pit_stops = pit_stops.merge(results[['raceId', 'driverId', 'constructorId', 'year']], 
                            on=['raceId', 'driverId'],how='left')
# Shape check: expect 11,371 rows, same as before the merge
print(pit_stops.shape)

# Team check: every pit stop should have found its team, expect 0
print(pit_stops['constructorId'].isna().sum())

(11371, 9)
0


In [25]:
# Deliberate exclusion: stops of 60+ seconds are race stoppage parking events
# (red flag or safety), not crew service. Documented data audit correction.
pit_stops_service = pit_stops[pit_stops['duration_seconds'] < 60].copy()

# Sanity check: expect roughly 517 rows fewer than before
print(pit_stops.shape, "->", pit_stops_service.shape)

(11371, 9) -> (10854, 9)


In [ ]:
# Pile step: average pit stop duration per team per season
# Uses pit_stops_service (crew service events only, 60s exclusion applied above)
# duration_seconds is fully populated, no null skipping happening here
avg_pit_stop = (pit_stops_service
    .groupby(['constructorId', 'year'])
    .agg(avg_pit_stop_duration=('duration_seconds', 'mean'))
    .reset_index()
)
avg_pit_stop.head()

,constructorId,year,avg_pit_stop_duration
0,1,2011,22.582109
1,1,2012,22.594636
2,1,2013,22.994317
3,1,2014,24.454975
4,1,2015,24.665944


In [29]:
print(avg_pit_stop.shape)                              # expect (147, 3)
print((avg_pit_stop['year'] == 2010).sum())            # expect 0
print(avg_pit_stop['avg_pit_stop_duration'].isna().sum())  # expect 0

(147, 3)
0
0
